# GridironIQ · Cap-spend EDA: finding (and fixing) the data traps

**Thesis:** in the salary-cap era, which positions actually win championships — and are teams paying for them correctly?

Before any of that holds up, the cap data has to be *right*. This notebook walks the two data-quality problems I hit building the Boardroom layer, and how the `gridiron` package fixes them:

1. **Overlapping contracts double-count the cap** — naive summation pushes a team's spend well past 400% of the cap.
2. **The source is thin before 2011** — which is why the models train on 2011–2024.

Everything imports from the package — the notebook *explores*, the module *owns the logic*.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from gridiron import config, db
from gridiron.ingestion import cap_data

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 30)

## 1. The raw source

nflverse ships OverTheCap's `historical_contracts` — **one row per contract a player has ever signed**, each with a projected multi-year cap schedule nested inside it.

In [ ]:
contracts = cap_data.load_raw_contracts()
print(f"{len(contracts):,} contracts, {contracts.shape[1]} columns")
cols = [c for c in ["player", "position", "team", "year_signed", "years", "value", "apy"]
        if c in contracts.columns]
contracts[cols].head()

## 2. Trap #1 — overlapping contracts double-count the cap

Every deal projects a *full* cap schedule, so when a player signs an extension the old deal and the new one **both** carry the transition seasons. Sum those naively and a team's cap balloons past 400%.

`gridiron` collapses this in one function — `_select_governing_contract` keeps, for each player-season, the most recently signed deal already in effect (`year_signed <= season`). To *show* what it fixes, I run the explode once with that de-dup disabled, then once normally.

In [ ]:
# NAIVE: temporarily disable the governing-contract de-dup
_orig = cap_data._select_governing_contract
cap_data._select_governing_contract = lambda df: df
naive = cap_data.build_player_year_caps(contracts)
cap_data._select_governing_contract = _orig          # restore

# FIXED: the real pipeline (also materialized as the player_year_caps table)
fixed = cap_data.build_player_year_caps(contracts)

print(f"exploded player-season rows  —  naive: {len(naive):,}   governing-only: {len(fixed):,}")

In [ ]:
def team_cap_pct(df):
    return (df.dropna(subset=["team"])
              .groupby(["season", "team"])["cap_percent"].sum()
              .mul(100))

naive_pct, fixed_pct = team_cap_pct(naive), team_cap_pct(fixed)
print(f"max team cap%  —  naive: {naive_pct.max():,.0f}%    fixed: {fixed_pct.max():.0f}%")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(naive_pct.clip(upper=500), bins=40, alpha=.55, label="naive (every deal)")
ax.hist(fixed_pct.clip(upper=500), bins=40, alpha=.85, label="governing contract only")
ax.axvline(100, color="k", ls="--", lw=1, label="100% of cap")
ax.set(xlabel="team cap spend (% of cap)", ylabel="team-seasons",
       title="De-duping overlapping contracts pulls spend back to ~100% of the cap")
ax.legend()
plt.tight_layout()

The naive tail runs into the hundreds of percent; the governing-contract view clusters right where it should — just under 100% of the cap. That one de-dup is the difference between garbage and a usable spending share.

## 3. Trap #2 — the source is thin before 2011

OverTheCap's coverage grows over time. Measuring *how much of each team's cap we can actually account for* (summed governing-contract share per team-season), the pre-2011 seasons are far too sparse to model honestly.

In [ ]:
coverage = fixed_pct.groupby("season").mean()   # mean % of cap accounted for, per team

fig, ax = plt.subplots(figsize=(9, 4))
coverage.plot(ax=ax, marker="o")
ax.axvspan(2011, config.END_SEASON, color="green", alpha=.08, label="modeling window")
ax.axvline(2011, color="red", ls="--", lw=1, label="2011 CBA")
ax.set(xlabel="season", ylabel="mean cap accounted for (%)",
       title="Cap coverage is sparse before 2011 -> models train on 2011-2024")
ax.legend()
plt.tight_layout()

Coverage before 2011 sits well below the 2011+ plateau, so the modeling layer (`cap_efficiency`, `ml_pipeline`, `clustering`) defaults to **2011–2024** and works in *normalized* within-team shares to stay era-comparable.

## 4. The cleaned result

With both traps handled, `build_positional_spending` produces sane positional shares. Here's a recent champion's spending profile straight from the pipeline's `positional_spending` table.

In [ ]:
pos = db.read_table("positional_spending")
kc23 = (pos[(pos.team == "KC") & (pos.season == 2023)]
        [["pos_group", "cap_pct", "cap_pct_norm", "n_players"]]
        .sort_values("cap_pct", ascending=False)
        .reset_index(drop=True))
kc23.assign(cap_pct=(kc23["cap_pct"] * 100).round(1),
            cap_pct_norm=(kc23["cap_pct_norm"] * 100).round(1))

## Takeaways

- **Governing-contract de-dup** is non-negotiable — without it a team's cap spend is nonsense (>400%). It lives in `cap_data._select_governing_contract`.
- The **2011–2024 window** falls straight out of the coverage curve, not a hunch.
- The notebook explores; the logic stays in `gridiron/` and is reused by the ETL, the models, and the Streamlit app.

Next: [`02_model_results.ipynb`](02_model_results.ipynb) — what the cleaned data says about paying to win.